## Práctica Optimización para Grandes Volúmenes de Datos: Clasificador ML de botnets
Alumnos:
- Jaime Vaquero Rabahieh 
- Pedro Arroyo Urbina
- Zakaria Lasry Sahraoui 
- Pablo Chicharro Gómez

In [19]:
from pyspark import SparkContext
import findspark
import numpy as np

sc = SparkContext("local[*]", "Ejercicio2")


26/03/07 16:34:23 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


## Ejercicio 2

### Ejecución sin Numpy en el normalize

In [20]:
def readFile (filename): 
    '''Arguments: 
    filename – name of the spam dataset file 
    12 columns: 11 features/dimensions (X) + 1 column with labels (Y) 
    Y -- Train labels (0 if normal traffic, 1 if botnet)  
    m rows: number of examples (m) 
    Returns: 
    An RDD containing the data of filename. Each example (row) of the file 
    corresponds to one RDD record. Each record of the RDD is a tuple (X,y).  
    “X” is an array containing the 11 features (float number) of an example  
    “y” is the 12th column of an example (integer 0/1) '''
    result = sc.textFile(filename)
    map_result = result.map(lambda row: [float(x) for x in row.split(",")])
    rdd_xy = map_result.map(lambda row: (row[:11],row[11]))
    return rdd_xy

In [21]:
# Prueba
data=readFile("./botnet_tot_syn_l.csv")
data.take(2)

[([9.012784269851089,
   1672.9999766891833,
   21.99998846107087,
   0.9999997452701503,
   61.99988768910407,
   69.99980788311223,
   13.000000232993186,
   2.9999999785934133,
   199.00000192395984,
   2468369573.0148935,
   2468372549.224571],
  1.0),
 ([3599.9990884099398,
   48206.57583515873,
   13362.999847123127,
   1.0000019205086303,
   262.99924185046257,
   82.99988759205826,
   13.999945225127506,
   5.000000032871429,
   216.99999612378573,
   1539044199.5873349,
   2468368394.7593513],
  0.0)]

In [22]:
rows_rdd = data.map(lambda line: line[0])
print(rows_rdd.take(1))
print()
cols_rdd = rows_rdd.flatMap(lambda row: [(i, (x,x*x, 1)) for i, x in enumerate(row)])
print(cols_rdd.take(5))
print()
group_rdd = cols_rdd.reduceByKey(lambda a,b:(a[0]+b[0],a[1]+b[1],a[2]+b[2]))
print(group_rdd.take(1))
print()
mean_rdd = group_rdd.map(lambda t: (t[0],(t[1][0] / t[1][2], np.sqrt((t[1][1] / t[1][2]) - (t[1][0] / t[1][2])**2))))
print(mean_rdd.collect())
print()
broadcast_var = sc.broadcast(dict(mean_rdd.collect()))

[[9.012784269851089, 1672.9999766891833, 21.99998846107087, 0.9999997452701503, 61.99988768910407, 69.99980788311223, 13.000000232993186, 2.9999999785934133, 199.00000192395984, 2468369573.0148935, 2468372549.224571]]

[(0, (9.012784269851089, 81.23028029487523, 1)), (1, (1672.9999766891833, 2798928.9220020077, 1)), (2, (21.99998846107087, 483.9994922872514, 1)), (3, (0.9999997452701503, 0.9999994905403655, 1)), (4, (61.99988768910407, 3843.9860734615186, 1))]



[Stage 3:======>                                                    (1 + 8) / 9]

[(0, (1281538272.9462628, 4221233394864.5664, 1000000))]

[(0, (1281.5382729462629, np.float64(1605.8932249182935))), (9, (2130603014.3773189, np.float64(713843442.8521906))), (1, (21282.767194882716, np.float64(24117.4750428853))), (10, (2261491802.491662, np.float64(1301531949.9426913))), (2, (6948.057181364392, np.float64(16394.945350993465))), (3, (62631.1960911871, np.float64(134264.21801130084))), (4, (122198594.92861378, np.float64(233920713.61972862))), (5, (15722236.101999493, np.float64(44123779.49184376))), (6, (9.107313932259263, np.float64(5.281725807207843))), (7, (1.8897547959110095, np.float64(2.10134561021457))), (8, (124.27015928413205, np.float64(90.3431222388102)))]



In [23]:
def normalize (RDD_Xy): 
    '''Arguments: 
    RDD_Xy is an RDD containing data examples. Each record of the RDD is a tuple 
    (X,y). 
    “X” is an array containing the 11 features (float number) of an example 
    “y” is the label of the example (integer 0/1)  
    Returns: 
    An RDD rescaled to N(0,1) in each column (mean=0, standard deviation=1) '''
    def map_normalize (RDD_Xy): 
        result = []
        x, y = RDD_Xy
        var = broadcast_var.value
        for i, x in enumerate(x):
             mean_aux, std_aux = var[i]
             if(std_aux!=0):
                 result.append((x - mean_aux)/std_aux)
             else:
                 result.append(0.0)
        return result, y
        
    rdd_norm = RDD_Xy.map(map_normalize)
    return rdd_norm

In [24]:
rdd_norm = normalize(data)
print(rdd_norm.take(2))

[([np.float64(-0.792409774778866), np.float64(-0.8130937083307339), np.float64(-0.4224507642218906), np.float64(-0.4664697491193844), np.float64(-0.5223929554497563), np.float64(-0.35631956924945285), np.float64(0.7370102960327225), np.float64(0.5283496333423401), np.float64(0.8271779941619601), np.float64(0.4731661571170488), np.float64(0.15895172357621995)], 1.0), ([np.float64(1.443720404002351), np.float64(1.1163610034798641), np.float64(0.39127563577819524), np.float64(-0.4664697491031832), np.float64(-0.5223920961870128), np.float64(-0.35631927462193325), np.float64(0.9263319360863808), np.float64(1.480120748267978), np.float64(1.0264183320400917), np.float64(-0.8286954523619166), np.float64(0.15894853159524694)], 0.0)]


In [25]:
import math
x_norm = rdd_norm.map(lambda xy: xy[0])

check = (
    x_norm
    .flatMap(lambda row: [(i, (v, v*v, 1)) for i, v in enumerate(row)])
    .reduceByKey(lambda a,b: (a[0]+b[0], a[1]+b[1], a[2]+b[2]))
    .mapValues(lambda t: (
        t[0]/t[2],  # mean
        math.sqrt(max((t[1]/t[2]) - (t[0]/t[2])**2, 0.0))  # std
    ))
    .collect()
)

print(sorted(check, key=lambda x: x[0]))

[Stage 10:======>                                                   (1 + 8) / 9]

[(0, (np.float64(2.1223843305051558e-13), 1.0000000000002456)), (1, (np.float64(1.5998016067442222e-13), 0.9999999999997955)), (2, (np.float64(-1.5675792042202375e-13), 1.0000000000001086)), (3, (np.float64(-3.327419051402103e-14), 1.0000000000005203)), (4, (np.float64(-1.8289730974174744e-13), 1.000000000000649)), (5, (np.float64(3.3446667657699438e-15), 1.000000000000576)), (6, (np.float64(7.27877541351063e-13), 1.0000000000005016)), (7, (np.float64(1.207932442071069e-12), 0.999999999998659)), (8, (np.float64(-3.5423472866114026e-13), 1.000000000001496)), (9, (np.float64(4.06921856210829e-13), 1.0000000000008586)), (10, (np.float64(-1.776409703779791e-13), 1.000000000000181))]


In [26]:
import numpy as np
import math

def sigmoid(z):
    if z >= 0:
        ez = math.exp(-z)
        return 1.0 / (1.0 + ez)
    else:
        ez = math.exp(z)
        return ez / (1.0 + ez)

def _sample_grad(xy, w, b):
    X, y = xy
    X = np.asarray(X, dtype=np.float64)
    y = float(y)

    z = float(np.dot(w, X) + b)
    y_hat = sigmoid(z)
    diff = y_hat - y

    grad_w = diff * X      # vector (11,)
    grad_b = diff          # escalar

    loss = -(y * math.log(y_hat)+(1-y)*math.log(1-y_hat))
    return (grad_w, grad_b, loss)

def train(RDD_Xy, iterations, learning_rate, lambda_reg):
    '''Arguments: 
    RDD_Xy --- RDD containing data examples. Each record of the RDD is a tuple 
    (X,y). 
    “X” is an array containing the 11 features (float number) of an example 
    “y” is the label of the example (integer 0/1)  
    iterations -- number of iterations of the optimization loop 
    learning_rate -- learning rate of the gradient descent 
    lambda_reg – regularization rate: l2 es el que vamos a aplicar
    
    Returns: 
    A list or array containing the weights “w” and bias “b”	at the end of the 
    training process'''	

    sc = RDD_Xy.context
    
    data = RDD_Xy.cache()
    m = data.count()
    if m == 0:
        raise ValueError("RDD_Xy vacío")

    k = len(data.first()[0])  # 11
    # inicialización
    rng = np.random.default_rng(42)
    w = rng.normal(0, 0.01, size=k).astype(np.float64)
    b = float(rng.normal(0, 0.01))
    for i in range(iterations):
        bc_w = sc.broadcast(w)
        bc_b = sc.broadcast(b)

        # suma de gradientes por todo el dataset
        sum_grad_w, sum_grad_b, sum_loss = data.map(
            lambda xy: _sample_grad(xy, bc_w.value, bc_b.value)
        ).reduce(
            lambda a, c: (a[0] + c[0],a[1] + c[1], a[2] + c[2])
        )

        bc_w.unpersist()
        bc_b.unpersist()

        # promedio + L2
        grad_w = (sum_grad_w / m) + (lambda_reg / k) * w 
        grad_b = (sum_grad_b / m)
        
        # update
        w = w - learning_rate * grad_w
        b = b - learning_rate * grad_b
        reg_term = (lambda_reg / (2*k)) * float(np.dot(w,w))
        J = (sum_loss / m ) + reg_term
        
        print(f"Loss en iteracion {i}: {J}")

    return [w, b]

In [27]:
def accuracy (w, b, RDD_Xy): 
    '''Arguments: 
    w -- weights 
    b -- bias 
    RDD_Xy – RDD containing examples to be predicted  
    Returns: 
    accuracy -- the number of predictions that are correct divided by the number         
    of records (examples) in RDD_xy.  
    Predict function can be used for predicting a single example'''
    pred_ok = RDD_Xy.map(
            lambda xy: 1 if predict(w, b, xy[0]) == int(xy[1]) else 0
        )
    
    correct = pred_ok.reduce(lambda a, c: a + c)
    total = RDD_Xy.count()
    return correct / total if total > 0 else 0.0

In [28]:
def predict (w, b, X): 
    '''Arguments: 
    w -- weights 
    b -- bias 
    X – Example to be predicted  
     
    Returns: 
    Y_pred – a value (0/1) corresponding to the prediction of X '''
    threshold=0.5
    z = float(np.dot(np.asarray(w, dtype=float), np.asarray(X, dtype=float)) + float(b))
    p = sigmoid(z)
    return 1 if p >= threshold else 0

#### Funciones nuevas para el ejercicio 2

Para aplicar el cross validation, hemos seguido los siguientes pasos:
- Primero lo que hacemos es asignar un número aleatorio entre 0 y *num_blocks_cv* a cada fila del RDD mediante un `map`, de tal manera que se genera una tupla en donde la clave es el número aleatorio y el valor es el rdd.
- A continuación, utilizando `flatMap`, lo que hacemos es crear dos rdds nuevos para seleccionar aquellas filas que van a test y aquellas que van a train. Si la i coincide con la clave de la fila entonces ira al rdd de test y si no coincide irá al rdd de train. 

In [29]:
def transform(RDD_Xy, num_blocks_cv):
    rng = np.random.default_rng(42)
    RDD_tranformed = RDD_Xy.map(lambda x: (rng.integers(0,num_blocks_cv), x))
    return RDD_tranformed

In [30]:
def get_block_data(data_cv,i):
    test_rdd = data_cv.flatMap(lambda x: [x[1]] if(x[0] == i) else [])
    train_rdd = data_cv.flatMap(lambda x: [x[1]] if(x[0] != i) else [])
    return train_rdd,test_rdd

In [31]:
nIter = 10
learningRate=1.5
lambda_reg=0

In [32]:
# standarize
data = normalize(data)
num_blocks_cv=10

# shuffle rows and transform data, specifying the number of blocks
data_cv = transform(data,num_blocks_cv)
data_cv.cache()
acc_tot = []
for i in range(num_blocks_cv):
    tr_data,test_data=get_block_data(data_cv,i)

    ws = train(tr_data,nIter,learningRate,lambda_reg)
    w,b = ws
    acc = accuracy(w,b,test_data)
    print("acc:",acc)
    acc_tot.append(acc)
    
avg_acc = sum(acc_tot) / len(acc_tot) if acc_tot else 0.0
print("average acc:", avg_acc)

26/03/07 16:34:34 WARN BlockManager: Task 59 already completed, not releasing lock for rdd_18_0


Loss en iteracion 0: 0.6908887570523111
Loss en iteracion 1: 0.3978280567170458
Loss en iteracion 2: 0.3286247248884867
Loss en iteracion 3: 0.29284707042462144
Loss en iteracion 4: 0.2700542231911392
Loss en iteracion 5: 0.25397693759387074
Loss en iteracion 6: 0.24191883789087146
Loss en iteracion 7: 0.2324924368639258
Loss en iteracion 8: 0.22489820345249525
Loss en iteracion 9: 0.21863785987250292
acc: 0.928446208717029


26/03/07 16:34:41 WARN BlockManager: Task 177 already completed, not releasing lock for rdd_33_0


Loss en iteracion 0: 0.6908799419472503
Loss en iteracion 1: 0.3977713292630002
Loss en iteracion 2: 0.3285790773162732
Loss en iteracion 3: 0.2928120084499007
Loss en iteracion 4: 0.2700283353703792
Loss en iteracion 5: 0.25395897008050444
Loss en iteracion 6: 0.24190772142893416
Loss en iteracion 7: 0.2324872613479039
Loss en iteracion 8: 0.22489819424181295
Loss en iteracion 9: 0.21864235730627005
acc: 0.9295245498121518


26/03/07 16:34:49 WARN BlockManager: Task 295 already completed, not releasing lock for rdd_48_0


...

Loss en iteracion 0: 0.6908985780480412
Loss en iteracion 1: 0.39807680914962323
Loss en iteracion 2: 0.3289280809302376
Loss en iteracion 3: 0.29316516071890153
Loss en iteracion 4: 0.27037720425113143
Loss en iteracion 5: 0.25430173240293197
Loss en iteracion 6: 0.24224438200388745
Loss en iteracion 7: 0.23281835172301468
Loss en iteracion 8: 0.22522435883241643
Loss en iteracion 9: 0.21896421571341945
acc: 0.930285882989716


26/03/07 16:35:33 WARN BlockManager: Task 1003 already completed, not releasing lock for rdd_138_0


Loss en iteracion 0: 0.6908948361624102
Loss en iteracion 1: 0.3978857467464235
Loss en iteracion 2: 0.32877336727597556
Loss en iteracion 3: 0.2930392863370023
Loss en iteracion 4: 0.2702725843615794
Loss en iteracion 5: 0.2542132878120391
Loss en iteracion 6: 0.24216870948435684
Loss en iteracion 7: 0.23275309218404672
Loss en iteracion 8: 0.22516781700614152
Loss en iteracion 9: 0.2189151324427256
acc: 0.9297017436062733


26/03/07 16:35:40 WARN BlockManager: Task 1121 already completed, not releasing lock for rdd_153_0


Loss en iteracion 0: 0.6908921697749094
Loss en iteracion 1: 0.3976486690783873
Loss en iteracion 2: 0.32848156055990424
Loss en iteracion 3: 0.2927317399829372
Loss en iteracion 4: 0.2699601132021867
Loss en iteracion 5: 0.253899455149164
Loss en iteracion 6: 0.24185472350606252
Loss en iteracion 7: 0.23243930455527148
Loss en iteracion 8: 0.2248542558690094
Loss en iteracion 9: 0.2186017088141531
acc: 0.9292052218276396
average acc: 0.929440740234706


In [33]:
sc.stop()

### Ejecución con Numpy en el normalize

In [34]:
sc = SparkContext("local[*]", "Ejercicio2")

26/03/07 16:35:46 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [35]:
import numpy as np
def normalize_numpy(data): 
    N = data.count()
    sum_x, sum_sq_x = data.map(
        lambda line: (np.array(line[0]), np.array(line[0])**2)
    ).reduce(
        lambda a, b: (a[0] + b[0], a[1] + b[1])
    )
    mean = sum_x / N
    # Gestión de la varianza negativa por errores numéricos
    variance = np.maximum((sum_sq_x / N) - (mean**2), 0.0)
    std_dev = np.sqrt(variance)
    std_dev[std_dev == 0] = 1.0  # Evitar división por cero
    return data.map(lambda x: ((np.array(x[0]) - mean) / std_dev, x[1]))

In [36]:
# read data
data=readFile("./botnet_tot_syn_l.csv")

# standarize
data = normalize_numpy(data)
num_blocks_cv=10

# shuffle rows and transform data, specifying the number of blocks
data_cv = transform(data,num_blocks_cv)
data_cv.cache()
acc_tot = []
for i in range(num_blocks_cv):
    tr_data,test_data=get_block_data(data_cv,i)

    ws = train(tr_data,nIter,learningRate,lambda_reg)
    w,b = ws
    acc = accuracy(w,b,test_data)
    print("acc:",acc)
    acc_tot.append(acc)
    
avg_acc = sum(acc_tot) / len(acc_tot) if acc_tot else 0.0
print("average acc:", avg_acc)

26/03/07 16:35:49 WARN BlockManager: Task 27 already completed, not releasing lock for rdd_5_0


Loss en iteracion 0: 0.6908887570523111
Loss en iteracion 1: 0.3978280567170458
Loss en iteracion 2: 0.3286247248884867
Loss en iteracion 3: 0.29284707042462144
Loss en iteracion 4: 0.2700542231911392
Loss en iteracion 5: 0.25397693759387074
Loss en iteracion 6: 0.24191883789087146
Loss en iteracion 7: 0.2324924368639258
Loss en iteracion 8: 0.22489820345249525
Loss en iteracion 9: 0.21863785987250292
acc: 0.928446208717029


26/03/07 16:35:53 WARN BlockManager: Task 145 already completed, not releasing lock for rdd_20_0


Loss en iteracion 0: 0.6908799419472503
Loss en iteracion 1: 0.3977713292630002
Loss en iteracion 2: 0.3285790773162732
Loss en iteracion 3: 0.2928120084499007
Loss en iteracion 4: 0.2700283353703792
Loss en iteracion 5: 0.25395897008050444
Loss en iteracion 6: 0.24190772142893416
Loss en iteracion 7: 0.2324872613479039
Loss en iteracion 8: 0.22489819424181295
Loss en iteracion 9: 0.21864235730627005
acc: 0.9295245498121518


26/03/07 16:35:58 WARN BlockManager: Task 263 already completed, not releasing lock for rdd_35_0


...

Loss en iteracion 0: 0.6908985780480412
Loss en iteracion 1: 0.39807680914962323
Loss en iteracion 2: 0.3289280809302376
Loss en iteracion 3: 0.29316516071890153
Loss en iteracion 4: 0.27037720425113143
Loss en iteracion 5: 0.25430173240293197
Loss en iteracion 6: 0.24224438200388745
Loss en iteracion 7: 0.23281835172301468
Loss en iteracion 8: 0.22522435883241643
Loss en iteracion 9: 0.21896421571341945
acc: 0.930285882989716


26/03/07 16:36:23 WARN BlockManager: Task 971 already completed, not releasing lock for rdd_125_0


Loss en iteracion 0: 0.6908948361624102
Loss en iteracion 1: 0.3978857467464235
Loss en iteracion 2: 0.32877336727597556
Loss en iteracion 3: 0.2930392863370023
Loss en iteracion 4: 0.2702725843615794
Loss en iteracion 5: 0.2542132878120391
Loss en iteracion 6: 0.24216870948435684
Loss en iteracion 7: 0.23275309218404672
Loss en iteracion 8: 0.22516781700614152
Loss en iteracion 9: 0.2189151324427256
acc: 0.9297017436062733


26/03/07 16:36:27 WARN BlockManager: Task 1089 already completed, not releasing lock for rdd_140_0


Loss en iteracion 0: 0.6908921697749094
Loss en iteracion 1: 0.3976486690783873
Loss en iteracion 2: 0.32848156055990424
Loss en iteracion 3: 0.2927317399829372
Loss en iteracion 4: 0.2699601132021867
Loss en iteracion 5: 0.253899455149164
Loss en iteracion 6: 0.24185472350606252
Loss en iteracion 7: 0.23243930455527148
Loss en iteracion 8: 0.2248542558690094
Loss en iteracion 9: 0.2186017088141531
acc: 0.9292052218276396
average acc: 0.929440740234706


In [37]:
sc.stop()